In [1]:
import os
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
client = chromadb.PersistentClient(path="../chroma_db")

collection = client.get_or_create_collection(
    name="semantic_documents"
)

print("ChromaDB connected successfully!")
print("Total documents:", collection.count())

ChromaDB connected successfully!
Total documents: 10


In [3]:
queries = pd.read_csv("../dataset/queries.csv")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Queries loaded successfully!")
print("Total queries:", len(queries))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Queries loaded successfully!
Total queries: 15


In [4]:
search_results = []
similarity_scores = []

for _, row in queries.iterrows():

    query = row["query"]

    query_embedding = model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    for i in range(len(results["ids"][0])):

        search_results.append({
            "query": query,
            "document": results["ids"][0][i],
            "content": results["documents"][0][i]
        })

        similarity_scores.append({
            "query": query,
            "document": results["ids"][0][i],
            "distance": results["distances"][0][i]
        })

print("Semantic search completed!")
print("Total retrieved results:", len(search_results))

Semantic search completed!
Total retrieved results: 75


In [5]:
pd.DataFrame(search_results).to_csv(
    "../dataset/search_results.csv",
    index=False
)

pd.DataFrame(similarity_scores).to_csv(
    "../dataset/similarity_scores.csv",
    index=False
)

print("Search results saved successfully!")

Search results saved successfully!


In [6]:
display(
    pd.DataFrame(search_results).groupby("query").head(5)
)

,query,document,content
0,How can computers perform tasks like humans?,document1.txt,Artificial Intelligence\n\nArtificial Intellig...
1,How can computers perform tasks like humans?,document6.txt,Natural Language Processing\n\nNatural Languag...
2,How can computers perform tasks like humans?,document2.txt,Machine Learning\n\nMachine Learning is a bran...
3,How can computers perform tasks like humans?,document10.txt,Internet of Things\n\nThe Internet of Things (...
4,How can computers perform tasks like humans?,document5.txt,Database Management System\n\nA Database Manag...
...,...,...,...
70,What technology connects smart home devices?,document10.txt,Internet of Things\n\nThe Internet of Things (...
71,What technology connects smart home devices?,document1.txt,Artificial Intelligence\n\nArtificial Intellig...
72,What technology connects smart home devices?,document8.txt,Cyber Security\n\nCyber security protects comp...
73,What technology connects smart home devices?,document3.txt,Deep Learning\n\nDeep Learning is a subset of ...


In [7]:
keyword_results = []

documents = []

for file in os.listdir("../dataset/documents"):
    if file.endswith(".txt"):
        with open("../dataset/documents/" + file, "r", encoding="utf-8") as f:
            documents.append({
                "document": file,
                "content": f.read()
            })

for _, row in queries.iterrows():
    query = row["query"].lower()
    keywords = query.split()

    scores = []

    for doc in documents:
        score = sum(
            1 for word in keywords
            if word in doc["content"].lower()
        )

        scores.append({
            "query": row["query"],
            "document": doc["document"],
            "keyword_score": score
        })

    scores.sort(key=lambda x: x["keyword_score"], reverse=True)
    keyword_results.extend(scores[:5])

print("Keyword search completed!")
print("Total results:", len(keyword_results))

Keyword search completed!
Total results: 75


In [8]:
keyword_df = pd.DataFrame(keyword_results)

keyword_df.to_csv(
    "../dataset/keyword_results.csv",
    index=False
)

print("Keyword results saved successfully!")

Keyword results saved successfully!


In [9]:
comparison = pd.merge(
    pd.DataFrame(search_results),
    keyword_df,
    on=["query", "document"],
    how="outer"
)

comparison.to_csv(
    "../dataset/response_comparison.csv",
    index=False
)

display(comparison.head(10))

,query,document,content,keyword_score
0,How can computers perform tasks like humans?,document1.txt,Artificial Intelligence\n\nArtificial Intellig...,3.0
1,How can computers perform tasks like humans?,document10.txt,Internet of Things\n\nThe Internet of Things (...,NaN
2,How can computers perform tasks like humans?,document2.txt,Machine Learning\n\nMachine Learning is a bran...,2.0
3,How can computers perform tasks like humans?,document3.txt,NaN,1.0
4,How can computers perform tasks like humans?,document5.txt,Database Management System\n\nA Database Manag...,NaN
5,How can computers perform tasks like humans?,document6.txt,Natural Language Processing\n\nNatural Languag...,1.0
6,How can computers perform tasks like humans?,document8.txt,NaN,1.0
7,How can computers recognize pictures?,document1.txt,Artificial Intelligence\n\nArtificial Intellig...,1.0
8,How can computers recognize pictures?,document2.txt,Machine Learning\n\nMachine Learning is a bran...,1.0
9,How can computers recognize pictures?,document3.txt,Deep Learning\n\nDeep Learning is a subset of ...,2.0


In [ ]:
print("SEMANTIC SEARCH VS KEYWORD SEARCH")
print("---------------------------------")
print("Total queries:", len(queries))
print("Semantic results:", len(search_results))
print("Keyword results:", len(keyword_results))

print("\nOBSERVATIONS:")
print("1. Semantic search understands the meaning of user queries.")
print("2. Keyword search mainly matches exact words.")
print("3. Semantic search can find related documents even when different words are used.")
print("4. Keyword search may miss relevant documents due to different wording.")
print("5. Semantic search improves information retrieval using vector embeddings.")